# ДЗ 3

In [48]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from math import comb, factorial

np.random.seed(42)

## Задача 1

Рассчитать с машинной точностью интеграл:
$$I = \int_{-1}^{1} \frac{\sin(x/2)}{e^x - 1}\,dx$$

квадратурой на разностных схемах.

Подынтегральная функция имеет устранимую особенность при $x=0$:

$$\frac{\sin(x/2)}{e^x-1} \to \frac{1}{2}$$

После устранения разрыва функция аналитична на $[-1, 1]$, поэтому применима теорема о квадратуре на разностных схемах:

$$\int_a^b f(x)\,dx = h \sum_{j=0}^{J-1}\sum_{k=-m}^{m} W_{m-k}^m\, y_{j+k} + O(h^{2m+2})$$

где $y_j = f(a + (j+0.5)\cdot h)$, $h = (b-a)/J$.

Отметим, что сетка смещена на $h/2$, поэтому при чётном $J$ точка $x=0$ не попадает в узлы.

In [49]:
def compute_D(n, j_max):
    D = [0.0] * (j_max + 1)
    D[0] = 1.0
    for j in range(j_max):
        val = 0.0
        p = n + 2 * j + 2
        for k in range(n + 2 * j + 1):
            for l in range(j + 1):
                bi = k - j + l
                bn = n + 2 * l
                if bi < 0 or bi > bn:
                    continue
                base = n + 2 * j - 2 * k
                val += (-1)**k * D[l] * comb(bn, bi) * base**p / (2**p * factorial(p))
        D[j + 1] = val
    return D

def compute_A(k, n, m_val):
    D = compute_D(n, m_val)
    val = 0.0
    for l in range(m_val + 1):
        bi = k - m_val + l
        bn = n + 2 * l
        if bi < 0 or bi > bn:
            continue
        val += (-1)**(k - m_val) * D[l] * comb(bn, bi)
    return val

def compute_W(m):
    W = np.zeros(2 * m + 1)
    for k in range(2 * m + 1):
        val = 0.0
        for n in range(m + 1):
            val += compute_A(k, 2 * n, m - n) / (2**(2 * n) * factorial(2 * n + 1))
        W[k] = val
    return W

def quadrature(f, a, b, J, m):
    h = (b - a) / J
    W = compute_W(m)

    y = {}
    for idx in range(-m, J + m):
        y[idx] = f(a + (idx + 0.5) * h)

    result = 0.0
    for j in range(J):
        for k in range(-m, m + 1):
            result += W[m - k] * y[j + k]

    return h * result

def integrand(x):
    if abs(x) < 1e-30:
        return 0.5
    return np.sin(x / 2.0) / np.expm1(x)

I_ref, _ = quad(integrand, -1, 1)
print(f"Scipy quad: {I_ref:.16f}")

m = 3
print(f"\nПорядок схемы m = {m}")
print()
for J in [4, 8, 16, 32, 64, 128]:
    I_val = quadrature(integrand, -1, 1, J, m)
    print(f"J = {J:>4} — I = {I_val:.16f},  |err| = {abs(I_val - I_ref):.2e}")

Scipy quad: 1.0130392362326262

Порядок схемы m = 3

J =    4 — I = 1.0130392166350386,  |err| = 1.96e-08
J =    8 — I = 1.0130392361530989,  |err| = 7.95e-11
J =   16 — I = 1.0130392362323128,  |err| = 3.13e-13
J =   32 — I = 1.0130392362326250,  |err| = 1.11e-15
J =   64 — I = 1.0130392362326264,  |err| = 2.22e-16
J =  128 — I = 1.0130392362326237,  |err| = 2.44e-15


## Задача 2

Рассчитать постоянную Эйлера-Маскерони с ошибкой в 13 знаке:
$$\gamma_E = \lim_{n \to \infty}\left(1 + \frac{1}{2} + \frac{1}{3} + \ldots + \frac{1}{n} - \ln n\right)$$

Представим $\gamma_E$ как сумму ряда $\gamma_E = \sum_{n=1}^{\infty} a_n$, где $a_n = \frac{1}{n} - \ln\!\left(1 + \frac{1}{n}\right)$.

Действительно, $\sum_{n=1}^{N} \ln\frac{n+1}{n} = \ln(N+1) \sim \ln N$, поэтому $\sum_{n=1}^{N} a_n = H_N - \ln(N+1) \to \gamma_E$.

По теореме о большой сумме плавной последовательности:
$$\sum_{j=1}^{\infty} a_j = \int_{j_m+1/2}^{\infty} a(x)\,dx + \sum_{j=1}^{j_m-m} a_j + \sum_{j=j_m-m+1}^{j_m+m}(1 - I_{j+m-j_m-1}^m)\, a_j + O(1/j_s^{2m+2})$$

Первое слагаемое вычисляется аналитически. Действительно, обозначив $c = j_m + 1/2$, получим:
$$\int_c^{\infty}\!\left(\frac{1}{x} - \ln\frac{x+1}{x}\right)dx = (c+1)\ln\frac{c+1}{c} - 1$$

Возьмем $m = 7$ и $j_m = 20$. Тогда ошибка $O(1/j_m^{16}) \sim 10^{-21}$, т.е. достигается необходимая точность

In [50]:
def compute_I_coeffs(m):
    W = compute_W(m)
    I = np.zeros(2 * m + 1)
    I[0] = W[2 * m]
    for l in range(1, 2 * m + 1):
        I[l] = I[l - 1] + W[2 * m - l]
    return I

def euler_mascheroni(j_m, m):
    I_coeffs = compute_I_coeffs(m)

    def a(j):
        return 1.0 / j - np.log1p(1.0 / j)

    c = j_m + 0.5
    tail = (c + 1) * np.log((c + 1) / c) - 1

    direct = sum(a(j) for j in range(1, j_m - m + 1))

    boundary = 0.0
    for j in range(j_m - m + 1, j_m + m + 1):
        l = j + m - j_m - 1
        boundary += (1 - I_coeffs[l]) * a(j)

    return tail + direct + boundary

gamma_exact = 0.57721566490153286060651209

m = 7
print(f"Порядок схемы m = {m}")
print()
for j_m in [10, 15, 20, 30, 50]:
    gamma = euler_mascheroni(j_m, m)
    print(f"j_m = {j_m:>3} — γ = {gamma:.16f},  |err| = {abs(gamma - gamma_exact):.2e}")

Порядок схемы m = 7

j_m =  10 — γ = 0.5772156648931934,  |err| = 8.34e-12
j_m =  15 — γ = 0.5772156649015259,  |err| = 6.99e-15
j_m =  20 — γ = 0.5772156649015338,  |err| = 8.88e-16
j_m =  30 — γ = 0.5772156649015341,  |err| = 1.22e-15
j_m =  50 — γ = 0.5772156649015338,  |err| = 8.88e-16


## Задача 3

Вычислить:
$$S = \sum_{n=1}^{\infty} \frac{n^2 + 1}{n^4 + n^2 + 1}\cos(2n)$$

с точностью $\varepsilon = 10^{-6}$.

На сайте показано, что можно посчитать следующим образом:

$$B = 1 - \pi + \frac{\pi^2}{6}$$

$$
S = B + \sum_{n=1}^{\infty} \frac{-\cos(2n)}{n^2(n^4+n^2+1)},
$$

причем для нужной точности достаточно 10 или более слагаемых

In [51]:
B = 1.0 - np.pi + np.pi**2 / 6.0

eps = 1e-6

S_remainder = 0.0
N_kummer = 0
for n in range(1, 1000):
    term = -np.cos(2 * n) / (n**2 * (n**4 + n**2 + 1))
    S_remainder += term
    if n > 1 and abs(term) < eps:
        N_kummer = n
        break

S_kummer = B + S_remainder

n=0
S_direct = 0.0
while True:
    n += 1
    term = (n**2 + 1) / (n**4 + n**2 + 1) * np.cos(2 * n)
    S_direct += term
    if n > 1 and abs(term) < eps:
        N_direct = n
        break

print(f"Метод Куммера: S = {S_kummer:.10f},  членов ряда: {N_kummer}")
print(f"Прямое суммирование: S = {S_direct:.10f},  членов ряда: {N_direct}")
print(f"Разница: = {abs(S_kummer - S_direct):.2e}")

Метод Куммера: S = -0.3512657640,  членов ряда: 10
Прямое суммирование: S = -0.3512801362,  членов ряда: 150
Разница: = 1.44e-05


## Задача 4

$$\lim_{x \to 0} \frac{y(x) - 1}{x}, \quad y(x) = \begin{cases} \cosh\sqrt{x}, & x > 0 \\ \cos\sqrt{|x|}, & x < 0 \end{cases}$$

Для начала найдем аналитическое значение и убедимся что оно конечное

При $x > 0$: $\cosh\sqrt{x} = 1 + \frac{x}{2} + \frac{x^2}{24} + \ldots \Rightarrow \frac{\cosh\sqrt{x} - 1}{x} = \frac{1}{2} + \frac{x}{24} + \ldots \to \frac{1}{2}$.

При $x < 0$: $\cos\sqrt{|x|} = 1 - \frac{|x|}{2} + \frac{|x|^2}{24} - \ldots = 1 + \frac{x}{2} + \frac{x^2}{24} + \ldots \Rightarrow \frac{\cos\sqrt{|x|} - 1}{x} = \frac{1}{2} + \frac{x}{24} + \ldots \to \frac{1}{2}$.

Предел равен $\frac{1}{2}$.

Теперь найдем функцию которую будем считать на сетке.

Обозначим $F(x) = (y(x) - 1)/x$. Симметризуем:
$$f(x) = \frac{F(x) + F(-x)}{2} = \frac{\cosh\sqrt{x} - \cos\sqrt{x}}{2x}, \quad x > 0$$

Получаем односторонний предел $x \to 0^+$. Тогда т.к. $\sqrt{x}$ вещественный сделаем замену $x = t^2$. Получаем двусторонний предел от
$$f(t^2) = \frac{\cosh(t) - \cos(t)}{2t^2} - \text{четная и аналитическая}$$

Далее по алгоритму строим сетку и т.д.

In [52]:
def f_symmetric(t):
    if abs(t) < 1e-30:
        return 0.5
    return (np.cosh(t) - np.cos(t)) / (2.0 * t**2)

def compute_limit(f_even, dx, m, N=None):
    if N is None:
        N = 2 * m + 1
    j0 = 2 * m + 2

    y = {}
    for k in range(j0):
        v = f_even((2 * k + 1) * dx / 2.0)
        y[k] = v
        y[-(k + 1)] = v

    result = 0.0
    for n in range(N + 1):
        nmod2 = n % 2
        upper_k = 2 * m + nmod2
        m_val = m - (n - nmod2) // 2
        coeff = (-1)**n / (4**n * factorial(n))
        for k in range(upper_k + 1):
            A_val = compute_A(k, n, m_val)
            y_idx = 2 * m + nmod2 - 2 * k
            result += A_val * coeff * y[y_idx]
    return result

exact = 0.5
m = 3

print(f"Порядок схемы m = {m}, число точек j0 = {2*m+2}")
print()
for dx in [2.0, 1.0, 0.5, 0.1, 0.01]:
    val = compute_limit(f_symmetric, dx, m)
    print(f"шаг = {dx} — предел = {val:.15f}")

Порядок схемы m = 3, число точек j0 = 8

шаг = 2.0 — предел = 0.902931007980118
шаг = 1.0 — предел = 0.500191957752235
шаг = 0.5 — предел = 0.500000579088194
шаг = 0.1 — предел = 0.500000000001456
шаг = 0.01 — предел = 0.499999999997539
